<a href="https://colab.research.google.com/github/yashverma1571/Yash-Verma-2026B15410163/blob/main/Yash_Verma_2026B15410163.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 12: Gym & Fitness Center Membership Analytics
**Industry:** Health & Wellness / Fitness  
**Dataset:** `12_gym_membership.csv`  
**Goal:** Evaluate member check-in logs, class attendance and membership tiers to improve facility scheduling and member retention.

> All results are shown as tables / printed output only (no charts, no matplotlib).

## Task 1: Load and inspect the dataset

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

# ---- Load the file ----
# In Google Colab: run this cell and use the "Choose Files" button to upload 12_gym_membership.csv
# (If you already uploaded it via the left Files panel, it is picked up automatically.)
import os
FILE = '12_gym_membership.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Place 12_gym_membership.csv next to this notebook.')

df = pd.read_csv(FILE)
print('Shape (rows, columns):', df.shape)
df.head()

Shape (rows, columns): (14730, 9)


,Checkin_ID,Member_ID,Gender,Age,Membership_Tier,Monthly_Fee,Class_Type,Checkin_Time,Workout_Duration_Mins
0,C111861,M1289,Male,30,Basic,499,Strength Training,2026-09-03 18:25:00,77.0
1,C102750,M1073,Male,22,Gold,999,CrossFit,2026-02-26 06:44:00,71.0
2,C102646,M1046,Male,38,Basic,499,CrossFit,2026-02-24 07:55:00,55.0
3,C101783,M1201,Female,31,Platinum,1499,Strength Training,2026-02-07 13:25:00,83.0
4,C104000,M1118,Male,28,Basic,499,HIIT,2026-03-24 13:06:00,55.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14730 entries, 0 to 14729
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Checkin_ID             14730 non-null  object 
 1   Member_ID              14730 non-null  object 
 2   Gender                 14730 non-null  object 
 3   Age                    14730 non-null  int64  
 4   Membership_Tier        14730 non-null  object 
 5   Monthly_Fee            14730 non-null  int64  
 6   Class_Type             14730 non-null  object 
 7   Checkin_Time           14730 non-null  object 
 8   Workout_Duration_Mins  13993 non-null  float64
dtypes: float64(1), int64(2), object(6)
memory usage: 1.0+ MB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Checkin_ID,14730,14700,C112149,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Member_ID,14730,299,M1126,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,14730,2,Male,8260,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,14730.0,NaN,NaN,NaN,40.85336,13.197701,18.0,28.0,42.0,52.0,62.0
Membership_Tier,14730,3,Gold,5718,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Monthly_Fee,14730.0,NaN,NaN,NaN,1009.726409,390.958409,499.0,499.0,999.0,1499.0,1499.0
Class_Type,14730,8,HIIT,2450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Checkin_Time,14730,14271,2026-02-14 19:04:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Workout_Duration_Mins,13993.0,NaN,NaN,NaN,58.577646,17.727118,3.0,46.0,58.0,70.0,236.0


## Task 2: Check missing values and duplicate `Checkin_ID` entries

In [4]:
print('Missing values per column:')
print(df.isna().sum())
print()
print('Missing %:')
print((df.isna().mean() * 100).round(2))

dup_ids = df['Checkin_ID'].duplicated().sum()
full_dups = df.duplicated().sum()
print(f'\nDuplicate Checkin_ID entries (extra rows): {dup_ids}')
print(f'Fully identical duplicate rows           : {full_dups}')
df[df['Checkin_ID'].duplicated(keep=False)].sort_values('Checkin_ID').head(10)

Missing values per column:
Checkin_ID                 0
Member_ID                  0
Gender                     0
Age                        0
Membership_Tier            0
Monthly_Fee                0
Class_Type                 0
Checkin_Time               0
Workout_Duration_Mins    737
dtype: int64

Missing %:
Checkin_ID               0.0
Member_ID                0.0
Gender                   0.0
Age                      0.0
Membership_Tier          0.0
Monthly_Fee              0.0
Class_Type               0.0
Checkin_Time             0.0
Workout_Duration_Mins    5.0
dtype: float64

Duplicate Checkin_ID entries (extra rows): 30
Fully identical duplicate rows           : 30


,Checkin_ID,Member_ID,Gender,Age,Membership_Tier,Monthly_Fee,Class_Type,Checkin_Time,Workout_Duration_Mins
859,C102200,M1108,Male,39,Gold,999,Pilates,2026-02-15 18:46:00,73.0
9658,C102200,M1108,Male,39,Gold,999,Pilates,2026-02-15 18:46:00,73.0
9531,C102223,M1033,Male,35,Basic,499,Pilates,2026-02-16 09:16:00,52.0
6828,C102223,M1033,Male,35,Basic,499,Pilates,2026-02-16 09:16:00,52.0
1955,C102697,M1278,Female,42,Gold,999,HIIT,2026-02-24 20:00:00,NaN
8734,C102697,M1278,Female,42,Gold,999,HIIT,2026-02-24 20:00:00,NaN
5265,C103481,M1051,Male,47,Platinum,1499,Open Gym,2026-03-13 17:21:00,81.0
9326,C103481,M1051,Male,47,Platinum,1499,Open Gym,2026-03-13 17:21:00,81.0
13766,C103531,M1083,Male,61,Gold,999,CrossFit,2026-03-14 19:02:00,74.0
2462,C103531,M1083,Male,61,Gold,999,CrossFit,2026-03-14 19:02:00,74.0


**Observation:** `Workout_Duration_Mins` has missing values and some `Checkin_ID`s appear more than once (exact repeated rows). Duplicates are removed first so they do not distort counts or the imputation values.

In [5]:
# Remove duplicate check-ins (keep the first occurrence of every Checkin_ID)
df = df.drop_duplicates(subset='Checkin_ID', keep='first').reset_index(drop=True)
print('Rows after removing duplicates:', len(df))
print('Duplicate Checkin_IDs left    :', df['Checkin_ID'].duplicated().sum())

Rows after removing duplicates: 14700
Duplicate Checkin_IDs left    : 0


## Task 3: Clean missing `Workout_Duration_Mins` values
**Chosen method: median imputation grouped by `Class_Type`.**

*Why?* Workout length depends strongly on the class (a HIIT session is shorter than Open Gym). The median is robust to the very short / very long outliers, so it is safer than the mean. Grouping by class gives a more realistic fill value than a single global number. If a class had no valid values, the global median is the fallback.

In [6]:
missing_before = df['Workout_Duration_Mins'].isna().sum()

class_median = df.groupby('Class_Type')['Workout_Duration_Mins'].transform('median')
df['Workout_Duration_Mins'] = df['Workout_Duration_Mins'].fillna(class_median)
df['Workout_Duration_Mins'] = df['Workout_Duration_Mins'].fillna(df['Workout_Duration_Mins'].median())

print('Missing before:', missing_before)
print('Missing after :', df['Workout_Duration_Mins'].isna().sum())

Missing before: 735
Missing after : 0


## Task 4: Convert `Checkin_Time` to datetime; extract weekday and hour

In [7]:
df['Checkin_Time'] = pd.to_datetime(df['Checkin_Time'])
df['Weekday'] = df['Checkin_Time'].dt.day_name()
df['Hour'] = df['Checkin_Time'].dt.hour
df['Month'] = df['Checkin_Time'].dt.to_period('M')

print(df['Checkin_Time'].dtype)
df[['Checkin_ID', 'Checkin_Time', 'Weekday', 'Hour', 'Month']].head()

datetime64[ns]


,Checkin_ID,Checkin_Time,Weekday,Hour,Month
0,C111861,2026-09-03 18:25:00,Thursday,18,2026-09
1,C102750,2026-02-26 06:44:00,Thursday,6,2026-02
2,C102646,2026-02-24 07:55:00,Tuesday,7,2026-02
3,C101783,2026-02-07 13:25:00,Saturday,13,2026-02
4,C104000,2026-03-24 13:06:00,Tuesday,13,2026-03


## Task 5: Total check-ins and average workout duration

In [8]:
total_checkins = len(df)
avg_duration = df['Workout_Duration_Mins'].mean()
print(f'Total check-ins        : {total_checkins}')
print(f'Unique members          : {df["Member_ID"].nunique()}')
print(f'Average workout duration: {avg_duration:.2f} minutes')

Total check-ins        : 14700
Unique members          : 299
Average workout duration: 58.55 minutes


## Task 6: Compare visit frequency across membership tiers

In [9]:
tier_summary = df.groupby('Membership_Tier').agg(
    Total_Checkins=('Checkin_ID', 'count'),
    Unique_Members=('Member_ID', 'nunique'),
    Avg_Duration_Mins=('Workout_Duration_Mins', 'mean')
)
tier_summary['Visits_per_Member'] = tier_summary['Total_Checkins'] / tier_summary['Unique_Members']
n_months = df['Month'].nunique()
tier_summary['Visits_per_Member_per_Month'] = tier_summary['Visits_per_Member'] / n_months
tier_summary = tier_summary.round(2).reindex(['Basic', 'Gold', 'Platinum'])
tier_summary

,Total_Checkins,Unique_Members,Avg_Duration_Mins,Visits_per_Member,Visits_per_Member_per_Month
Membership_Tier,,,,,
Basic,4338,134,58.65,32.37,3.24
Gold,5707,108,58.32,52.84,5.28
Platinum,4655,57,58.73,81.67,8.17


## Task 7: Peak gym hours and busiest days of the week

In [10]:
hour_counts = df.groupby('Hour').size().rename('Checkins').to_frame()
hour_counts['Share_%'] = (hour_counts['Checkins'] / hour_counts['Checkins'].sum() * 100).round(2)
print('Top 5 peak hours:')
print(hour_counts.sort_values('Checkins', ascending=False).head(5))

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['Weekday'].value_counts().reindex(day_order).rename('Checkins').to_frame()
day_counts['Share_%'] = (day_counts['Checkins'] / day_counts['Checkins'].sum() * 100).round(2)
print('\nCheck-ins by day of week:')
print(day_counts)
print('\nBusiest day:', day_counts['Checkins'].idxmax())

Top 5 peak hours:
      Checkins  Share_%
Hour                   
18        1701    11.57
17        1495    10.17
7         1330     9.05
19        1315     8.95
16        1225     8.33

Check-ins by day of week:
           Checkins  Share_%
Weekday                     
Monday         2494    16.97
Tuesday        2491    16.95
Wednesday      2094    14.24
Thursday       2355    16.02
Friday         2001    13.61
Saturday       1703    11.59
Sunday         1562    10.63

Busiest day: Monday


In [11]:
# Peak-hour schedule summary: Weekday x Hour matrix (top view of busy slots)
schedule = pd.crosstab(df['Weekday'], df['Hour']).reindex(day_order)
print('Check-ins by weekday and hour:')
schedule

Check-ins by weekday and hour:


Hour,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
Weekday,,,,,,,,,,,,,,,,,,
Monday,78,167,212,173,112,85,85,102,109,69,102,209,261,272,217,141,73,27
Tuesday,73,171,236,166,92,83,78,109,109,73,93,202,272,298,192,145,73,26
Wednesday,61,116,183,139,87,65,59,74,97,86,81,161,212,266,200,116,65,26
Thursday,77,136,224,175,94,59,72,101,91,74,107,193,234,263,223,142,69,21
Friday,50,119,183,143,65,65,56,89,74,57,79,175,207,233,180,136,70,20
Saturday,48,105,152,130,73,48,53,60,65,35,79,147,158,192,159,111,67,21
Sunday,41,82,140,109,67,59,42,56,59,46,83,138,151,177,144,100,50,18


In [12]:
# Group hours into time slots for a compact schedule summary
slots = pd.cut(df['Hour'], bins=[4, 8, 11, 15, 18, 23],
               labels=['Early Morning (5-8)', 'Late Morning (9-11)', 'Afternoon (12-15)',
                       'Evening Peak (16-18)', 'Night (19-22)'])
slot_summary = df.groupby(slots, observed=True).size().rename('Checkins').to_frame()
slot_summary['Share_%'] = (slot_summary['Checkins'] / len(df) * 100).round(2)
slot_summary

,Checkins,Share_%
Hour,,
Early Morning (5-8),3689,25.10
Late Morning (9-11),1499,10.20
Afternoon (12-15),2259,15.37
Evening Peak (16-18),4421,30.07
Night (19-22),2832,19.27


## Task 8: Rank fitness classes by total attendance

In [13]:
class_rank = df['Class_Type'].value_counts().rename('Total_Attendance').to_frame()
class_rank['Share_%'] = (class_rank['Total_Attendance'] / len(df) * 100).round(2)
class_rank['Avg_Duration_Mins'] = df.groupby('Class_Type')['Workout_Duration_Mins'].mean().round(2)
class_rank.insert(0, 'Rank', range(1, len(class_rank) + 1))
class_rank

,Rank,Total_Attendance,Share_%,Avg_Duration_Mins
Class_Type,,,,
HIIT,1,2446,16.64,42.95
Strength Training,2,2186,14.87,72.51
Open Gym,3,2078,14.14,77.44
Yoga,4,2041,13.88,57.39
Zumba,5,1790,12.18,53.53
Spinning,6,1768,12.03,47.56
Pilates,7,1219,8.29,53.61
CrossFit,8,1172,7.97,62.93


## Task 9: Average workout duration — male vs female

In [14]:
gender_dur = df.groupby('Gender')['Workout_Duration_Mins'].agg(['count', 'mean', 'median']).round(2)
gender_dur.columns = ['Checkins', 'Mean_Duration', 'Median_Duration']
print(gender_dur)
diff = gender_dur.loc['Male', 'Mean_Duration'] - gender_dur.loc['Female', 'Mean_Duration']
print(f'\nMale members work out on average {diff:+.2f} minutes compared with female members.')

        Checkins  Mean_Duration  Median_Duration
Gender                                          
Female      6462          55.98             55.0
Male        8238          60.55             60.0

Male members work out on average +4.57 minutes compared with female members.


## Task 10: Monthly membership revenue by tier
**Assumption:** a member is *active* (and pays that tier's `Monthly_Fee`) in every month in which they have at least one check-in. Revenue = number of active members × monthly fee of the tier.

In [15]:
active = df.drop_duplicates(subset=['Member_ID', 'Month'])
revenue = (active.groupby(['Month', 'Membership_Tier'])['Monthly_Fee'].sum()
           .unstack(fill_value=0)[['Basic', 'Gold', 'Platinum']])
revenue['Total'] = revenue.sum(axis=1)
print('Monthly membership revenue by tier (INR):')
revenue

Monthly membership revenue by tier (INR):


Membership_Tier,Basic,Gold,Platinum,Total
Month,,,,
2026-01,54890,105894,85443,246227
2026-02,58383,101898,85443,245724
2026-03,60878,104895,83944,249717
2026-04,55888,104895,85443,246226
2026-05,57385,103896,85443,246724
2026-06,58383,100899,85443,244725
2026-07,53892,105894,85443,245229
2026-08,54391,105894,85443,245728
2026-09,54890,106893,85443,247226


In [16]:
print('Total revenue by tier over the whole period (INR):')
revenue.sum().to_frame('Revenue').assign(**{'Share_%': lambda x: (x['Revenue'] / x.loc['Total', 'Revenue'] * 100).round(2)})

Total revenue by tier over the whole period (INR):


,Revenue,Share_%
Membership_Tier,,
Basic,567862,23.00
Gold,1047951,42.45
Platinum,852931,34.55
Total,2468744,100.00


## Task 11: Members with fewer than 2 visits per month (low-engagement)
**Definition:** a member's average visits per month = total check-ins ÷ number of months in the dataset. If this is below 2, the member is flagged as low-engagement.

In [17]:
member_visits = df.groupby(['Member_ID', 'Membership_Tier']).size().rename('Total_Checkins').reset_index()
member_visits['Avg_Visits_per_Month'] = (member_visits['Total_Checkins'] / n_months).round(2)

low_engagement = member_visits[member_visits['Avg_Visits_per_Month'] < 2]
print('Months covered in data          :', n_months)
print('Total members                   :', len(member_visits))
print('Low-engagement members (<2/mo)  :', len(low_engagement))
print(f'Share of all members            : {len(low_engagement) / len(member_visits) * 100:.1f}%')
print('\nLow-engagement members by tier:')
print(low_engagement['Membership_Tier'].value_counts())
low_engagement.sort_values('Avg_Visits_per_Month').head(10)

Months covered in data          : 10
Total members                   : 299
Low-engagement members (<2/mo)  : 41
Share of all members            : 13.7%

Low-engagement members by tier:
Membership_Tier
Basic    36
Gold      5
Name: count, dtype: int64


,Member_ID,Membership_Tier,Total_Checkins,Avg_Visits_per_Month
23,M1023,Basic,2,0.2
71,M1071,Basic,2,0.2
228,M1229,Basic,2,0.2
141,M1141,Basic,2,0.2
292,M1293,Basic,2,0.2
109,M1109,Gold,3,0.3
59,M1059,Basic,3,0.3
138,M1138,Basic,3,0.3
229,M1230,Basic,3,0.3
168,M1168,Basic,3,0.3


## Task 12: NumPy — median, variance and quartiles of workout duration

In [18]:
dur = df['Workout_Duration_Mins'].to_numpy()

median = np.median(dur)
variance = np.var(dur, ddof=1)       # sample variance
std = np.std(dur, ddof=1)
q1, q2, q3 = np.percentile(dur, [25, 50, 75])

print(f'Median   : {median:.2f}')
print(f'Variance : {variance:.2f}')
print(f'Std. dev.: {std:.2f}')
print(f'Q1 (25%) : {q1:.2f}')
print(f'Q2 (50%) : {q2:.2f}')
print(f'Q3 (75%) : {q3:.2f}')

Median   : 57.00
Variance : 305.94
Std. dev.: 17.49
Q1 (25%) : 46.00
Q2 (50%) : 57.00
Q3 (75%) : 70.00


## Task 13: Identify unusually long / short sessions using IQR

In [19]:
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print(f'IQR = {iqr:.2f} | Lower fence = {lower_fence:.2f} | Upper fence = {upper_fence:.2f}')

df['IQR_Flag'] = np.select(
    [df['Workout_Duration_Mins'] < lower_fence, df['Workout_Duration_Mins'] > upper_fence],
    ['Unusually Short', 'Unusually Long'],
    default='Normal'
)
print()
print(df['IQR_Flag'].value_counts())
df[df['IQR_Flag'] != 'Normal'][['Checkin_ID', 'Member_ID', 'Class_Type', 'Workout_Duration_Mins', 'IQR_Flag']].head(10)

IQR = 24.00 | Lower fence = 10.00 | Upper fence = 106.00

IQR_Flag
Normal             14657
Unusually Long        32
Unusually Short       11
Name: count, dtype: int64


,Checkin_ID,Member_ID,Class_Type,Workout_Duration_Mins,IQR_Flag
257,C102393,M1076,Yoga,4.0,Unusually Short
699,C104319,M1050,Pilates,234.0,Unusually Long
722,C104414,M1128,Zumba,198.0,Unusually Long
865,C101015,M1073,Strength Training,109.0,Unusually Long
2259,C110678,M1252,Pilates,189.0,Unusually Long
2436,C101645,M1112,Open Gym,109.0,Unusually Long
2803,C113384,M1291,Strength Training,110.0,Unusually Long
2817,C101622,M1297,HIIT,177.0,Unusually Long
2975,C104987,M1211,HIIT,219.0,Unusually Long
3571,C111451,M1066,Open Gym,109.0,Unusually Long


## Task 14: Workout-intensity label (documented thresholds)
| Label | Rule (minutes) |
|---|---|
| **Short** | duration < 40 |
| **Moderate** | 40 ≤ duration ≤ 75 |
| **Extended** | duration > 75 |

Thresholds are practical gym-session lengths (under 40 min = quick session; over 75 min = long session).

In [20]:
df['Workout_Intensity'] = np.select(
    [df['Workout_Duration_Mins'] < 40,
     (df['Workout_Duration_Mins'] >= 40) & (df['Workout_Duration_Mins'] <= 75),
     df['Workout_Duration_Mins'] > 75],
    ['Short', 'Moderate', 'Extended'],
    default='Unknown'
)
intensity = df['Workout_Intensity'].value_counts().reindex(['Short', 'Moderate', 'Extended']).rename('Checkins').to_frame()
intensity['Share_%'] = (intensity['Checkins'] / len(df) * 100).round(2)
print(intensity)
print()
print('Intensity by membership tier (%):')
(pd.crosstab(df['Membership_Tier'], df['Workout_Intensity'], normalize='index') * 100).round(1)

                   Checkins  Share_%
Workout_Intensity                   
Short                  1902    12.94
Moderate              10263    69.82
Extended               2535    17.24

Intensity by membership tier (%):


Workout_Intensity,Extended,Moderate,Short
Membership_Tier,,,
Basic,17.6,69.5,12.8
Gold,16.8,70.1,13.1
Platinum,17.4,69.7,12.9


## Expected Deliverables — summary tables

In [21]:
print('=== 1. PEAK-HOUR SCHEDULE SUMMARY ===')
print(hour_counts.sort_values('Checkins', ascending=False).head(5))
print('\nBusiest day:', day_counts['Checkins'].idxmax(), '| Quietest day:', day_counts['Checkins'].idxmin())

print('\n=== 2. CLASS POPULARITY TABLE ===')
print(class_rank)

print('\n=== 3. TIER COMPARISON ===')
print(tier_summary)

print('\n=== 4. LOW-ENGAGEMENT MEMBER COUNT ===')
print(len(low_engagement), 'members (', round(len(low_engagement) / len(member_visits) * 100, 1), '% )')

=== 1. PEAK-HOUR SCHEDULE SUMMARY ===
      Checkins  Share_%
Hour                   
18        1701    11.57
17        1495    10.17
7         1330     9.05
19        1315     8.95
16        1225     8.33

Busiest day: Monday | Quietest day: Sunday

=== 2. CLASS POPULARITY TABLE ===
                   Rank  Total_Attendance  Share_%  Avg_Duration_Mins
Class_Type                                                           
HIIT                  1              2446    16.64              42.95
Strength Training     2              2186    14.87              72.51
Open Gym              3              2078    14.14              77.44
Yoga                  4              2041    13.88              57.39
Zumba                 5              1790    12.18              53.53
Spinning              6              1768    12.03              47.56
Pilates               7              1219     8.29              53.61
CrossFit              8              1172     7.97              62.93

=== 3. TIER CO

## Task 15: Five facility-management and retention recommendations

In [22]:
busiest_hours = hour_counts['Checkins'].nlargest(3).index.tolist()
quiet_day = day_counts['Checkins'].idxmin()
busy_day = day_counts['Checkins'].idxmax()
top_classes = class_rank.index[:3].tolist()
low_classes = class_rank.index[-2:].tolist()
low_n = len(low_engagement)

recs = [
 f"1. STAFFING & CROWD CONTROL: Peak hours are {busiest_hours}. Add trainers/floor staff and extra equipment slots in these hours, and schedule cleaning/maintenance in off-peak hours.",
 f"2. DAY-BASED PROMOTIONS: {busy_day} is the busiest day and {quiet_day} the quietest. Offer off-peak discounts or bonus classes on {quiet_day} to spread the load.",
 f"3. CLASS PORTFOLIO: Top classes are {top_classes}. Add more batches of these in peak slots; review timing or promotion of the lowest-attended classes {low_classes}.",
 f"4. RETENTION OUTREACH: {low_n} members visit fewer than 2 times a month. Send reminder messages, free personal-training intro sessions, or a 'comeback' offer before they cancel.",
 "5. TIER UPGRADE STRATEGY: Use tier visit data to target highly active Basic members with Gold/Platinum upgrade offers, and give Platinum members exclusive priority booking to justify the higher fee."
]
for r in recs:
    print(r, '\n')

1. STAFFING & CROWD CONTROL: Peak hours are [18, 17, 7]. Add trainers/floor staff and extra equipment slots in these hours, and schedule cleaning/maintenance in off-peak hours. 

2. DAY-BASED PROMOTIONS: Monday is the busiest day and Sunday the quietest. Offer off-peak discounts or bonus classes on Sunday to spread the load. 

3. CLASS PORTFOLIO: Top classes are ['HIIT', 'Strength Training', 'Open Gym']. Add more batches of these in peak slots; review timing or promotion of the lowest-attended classes ['Pilates', 'CrossFit']. 

4. RETENTION OUTREACH: 41 members visit fewer than 2 times a month. Send reminder messages, free personal-training intro sessions, or a 'comeback' offer before they cancel. 

5. TIER UPGRADE STRATEGY: Use tier visit data to target highly active Basic members with Gold/Platinum upgrade offers, and give Platinum members exclusive priority booking to justify the higher fee. 



## Conclusion
The analysis cleaned the data (duplicates removed, missing durations imputed by class median), derived time features, compared tiers, classes, genders and revenue, identified outlier sessions with IQR, and produced actionable recommendations for scheduling and retention — all without any plotting libraries.